In [ ]:
# Colab setup: Python 3.13 compatible SentencePiece. Run this cell first.
import importlib.util, subprocess, sys
print("Python:", sys.version.split()[0])
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", "transformers==4.49.0", "sentencepiece==0.2.1"])
print("Dependencies installed.")


In [ ]:
import json, time
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
torch.set_num_threads(2)
MODEL_ID = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, trust_remote_code=False, use_safetensors=True).to('cpu').eval()
context = 'You can check your application status using its reference number.'
settings = {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 48}


In [ ]:
question = 'How can I check my application status?'
prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
start = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(**inputs, **settings)
seconds = time.perf_counter() - start
answer = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
assert answer, 'The model returned an empty answer.'
result = {'mode':'real_local_model', 'model': MODEL_ID,           'device':'cpu', 'context':context, 'question':question, 'answer':answer,
          'generation':settings, 'input_tokens':int(inputs['input_ids'].shape[-1]),
          'output_tokens_including_special':int(generated.shape[-1]),
          'inference_seconds':round(seconds, 3), 'external_inference_api_calls':0,
          'semantic_review':'Verify that the answer directs users to their reference number, with no invented fees, dates or statuses.'}
Path('murshid_live_outputs').mkdir(exist_ok=True)
Path('murshid_live_outputs/live_demo_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
print(json.dumps(result, indent=2))


In [ ]:
# Explanation: The context and question are tokenized, then FLAN-T5 generates output tokens.
# Unlike FakeClient's dictionary lookup, generation uses learned model weights.
# A generated answer must still be reviewed for correctness.
print('The model generates token sequences; FakeClient returns a fixed hand-written response.')
